# Create Dataframes from the below Data 

## usecase -1

The data team has provided two CSV files — customer_data.csv and sales_data.csv. Load them into Spark DataFrames, infer schemas automatically, and print the schema along with the first 5 rows of each.

In [0]:
# create both Customer and transaction drataframes from the given source
#cust_schema="customer_id string,gender string,age int,payment_method string"
#sales_schema="invoice_no string,customer_id string,category string,quantity int,price decimal(10,2),invoice_date string,shopping_mall string"
cust_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/izwd37dev/wd37db/rawdatta/Use_case_30062026/customer_data.csv")
sales_df=spark.read.format("csv").option("header","true").load("/Volumes/izwd37dev/wd37db/rawdatta/Use_case_30062026/sales_data.csv")
cust_df.show(5)
sales_df.show(5)
cust_df.printSchema()
sales_df.printSchema()

## usecase- 2

The marketing team wants to know the total revenue generated by the 'Clothing' category for each month.

 Revenue = quantity × price.



In [0]:
#total revenue generated by clothing team
from pyspark.sql.functions import *
#sales_df.createOrReplaceTempView("sales_vw")
cust_df.createOrReplaceTempView("cust_vw")
sales_df_transformed=sales_df.withColumn("price",sales_df.price.cast("decimal(10,2)")).withColumn("quantity",sales_df.quantity.cast("int")).withColumn("invoice_date",to_date(sales_df.invoice_date,"dd-MM-yyyy"))
sales_df_transformed.show(5)
sales_df_transformed.printSchema()
sales_df_transformed.filter(upper(col("category")) == "CLOTHING").groupBy(month("invoice_date").alias("Month")).agg(sum(col("price") * col("quantity")).alias("Revenue"),sum(col("quantity")).alias("tot_quantity"),sum(col("price")).alias("tot_price")).orderBy("Month").display()
#spark.sql("select sum(price) from sales where category='Clothing'").show()
#total revenue generated by electronics team
sales_df_transformed.createOrReplaceTempView("sales_vw")
#spark.sql("select category, sum(price * quantity) as revenue, sum(price) tot_price, sum(quantity) tot_quantity from sales_vw where upper(category)='CLOTHING' group by category").show()
#total revenue generated by electronics team



## usecase 3

Who is the best customer? Find the customer who has spent the most money overall. Include their customer_id and name (from the customer table). Show the top 5.

In [0]:
print("***************************SQL******************************")
print("\n")
spark.sql("select c.customer_id, sum(cast(s.price as decimal)*cast(s.quantity as int)) as total_revenue, sum(cast(s.price as decimal)) total_price,sum(cast(s.quantity as int)) total_quantity from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by c.customer_id order by total_revenue desc,c.customer_id limit 5 ").show()
spark.sql("""
    SELECT 
        c.customer_id,
        SUM(CAST(s.price AS DECIMAL(18,2)) * CAST(s.quantity AS INT)) AS total_revenue,
        SUM(CAST(s.price AS DECIMAL(18,2))) AS total_price,
        SUM(CAST(s.quantity AS INT)) AS total_quantity
    FROM cust_vw c
    INNER JOIN sales_vw s 
        ON c.customer_id = s.customer_id
    GROUP BY c.customer_id
    ORDER BY total_revenue DESC, c.customer_id
    LIMIT 5
""").show()

print("\n")
print("***************************DSL******************************")
join_df=cust_df.alias("c").join(sales_df_transformed.alias("s"),col("c.customer_id")==col("s.customer_id"),"inner").groupBy("c.customer_id").agg(sum(col("s.price")*col("s.quantity")).alias("total_revenue"),sum(col("s.price")).alias("total_price"),sum(col("s.quantity")).alias("total_quantity")).orderBy(desc("total_revenue"),asc("c.customer_id")).limit(5)
join_df.show()

## usecase 4

Generate a customer revenue summary.

 For each customer, show their total revenue and classify them as
  'High Value' (≥ 5000), 
  'Mid Value' (≥ 1000), or 
  'Low Value' (< 1000)

In [0]:

print("***************************SQL******************************")
spark.sql("select c.customer_id, sum(s.price*s.quantity) total_revenue, case when sum(s.price*s.quantity) >= 5000 then 'High Value' when sum(s.price*s.quantity) >= 1000 then 'Mid Value'  else 'Low Value' end as Revenue_category from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by c.customer_id order by 1 ").show(5)

print("\n")
print("***************************DSL******************************")
cust_df.alias("c").join(sales_df_transformed.alias("s"),col("c.customer_id")==col("s.customer_id"),"inner").groupBy("c.customer_id").agg(sum(col("price")*col("quantity")).alias("total_revenue")).withColumn("Revenue_category",when(col("total_revenue")>=5000,"High Value").when(col("total_revenue")>=1000,"Mid Value").otherwise("Low Value")).orderBy("c.customer_id").show(5)


## usecase 5

Is there a significant spending difference between male and female customers? Compute average transaction value, total revenue, and transaction count by gender.

In [0]:
spark.sql("select gender,avg(cast(s.price as decimal(10,2))) average_transaction,sum(cast(s.price as decimal(10,2))*cast(s.quantity as int)) total_revenue_at_gender_level, count(1) transaction_count from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by gender ").show()

cust_df.alias("c").join(sales_df_transformed.alias("s"),col("c.customer_id")==col("s.customer_id"),"inner").groupBy("c.gender").agg(avg("price").alias("average_transactions"),sum(col("price") * col("quantity")).alias("total_revenue_at_gender_level"),count("*").alias("transaction_count")).show()

## usecase-6

The finance team wants to know which payment method is most popular for each product category. Show the count of transactions per payment method per category.

In [0]:
spark.sql("select category,payment_method,count(1) transaction_count from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by category,payment_method order by transaction_count desc").show(5)

cust_df.alias("c").join(sales_df_transformed.alias("s"),col("c.customer_id")==col("s.customer_id"),"inner").groupBy(col("category"),col("payment_method")).agg(count("*").alias("transaction_count")).orderBy(desc("transaction_count")).show(5)



## usecase 7

Data integrity check: Are there any transactions with a customer_id that does not exist in the customer master table? Identify and count orphaned transactions.


In [0]:
spark.sql("select s.customer_id,count(1) from sales_vw s left join cust_vw c on s.customer_id=c.customer_id where c.customer_id is null group by s.customer_id").show()
spark.sql("select s.customer_id,count(1) from sales_vw s left anti join cust_vw c on s.customer_id=c.customer_id group by s.customer_id").show()

sales_df_transformed.alias("c").join(cust_df.alias("s"),col("c.customer_id")==col("s.customer_id"),"left_anti").groupBy(col("c.customer_id")).count().show()

## usecase 8

The marketing team wants to target ads by age group. Bucket customers into: Teens (< 20), Young Adults (20–35), Adults (36–50), Seniors (50+). Which segment generates the most revenue?

In [0]:
from pyspark.sql.functions import  *
from pyspark.sql.window import  *
df_res8=spark.sql("select age_group,sum(price*quantity) Revenue from (select case when age<20 then 'Teenager' when age between 20 and 35 then 'Young Adults' when age between 36 and 50 then 'Adults' else 'Senior' end as age_group, price,quantity from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id )group by age_group order by 2 desc")

display(df_res8)
df_res8.withColumn("row_num",row_number().over(Window.orderBy(desc("Revenue")))).filter("row_num=1").show()

cust_df.alias("c").join(sales_df_transformed.alias("s"),col("c.customer_id")==col("s.customer_id"),"inner").withColumn("age_group",when(col("age") < 20, "Teenager").when(col("age").between(20,35), "Young Adults").when(col("age").between(36,50), "Adults").otherwise("Senior")).groupBy("age_group").agg(sum(col("price")*col("quantity")).alias("Revenue")).orderBy(desc("revenue")).show(5)


## usecase 9

The retention team wants to run a win-back campaign. Identify all customers whose most recent purchase is more than 90 days before the latest invoice date in the dataset.

consider max_date from the table as current_date

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *
spark.sql("""select s.customer_id,max(invoice_date) latest_invoice_date,current_date() 
from sales_vw s inner join cust_vw c on s.customer_id=c.customer_id where date_diff(current_date(),invoice_date)>90 group by s.customer_id """).show(2) 

#Find the latest invoice date in the dataset
# Using collect()
spark.sql("select max(invoice_date) as max_date from sales_vw").collect()[0]['max_date']
# Using first()
spark.sql("select max(invoice_date) as max_date from sales_vw").first()["max_date"]

# Using head()
spark.sql("select max(invoice_date) as max_date from sales_vw").head()["max_date"]


max_date = spark.sql("select max(invoice_date) as max_date from sales_vw").collect()[0]['max_date']

# Identify customers whose most recent purchase is more than 90 days before max_date
spark.sql(f"""
    select s.customer_id, max(s.invoice_date) as latest_invoice_date
    from sales_vw s
    inner join cust_vw c on s.customer_id = c.customer_id
    group by s.customer_id
    having datediff('{max_date}', max(s.invoice_date)) > 90
""").show()

# Find the latest invoice date in the dataset
max_date = sales_df_transformed.agg(max("invoice_date").alias("max_date")).collect()[0]["max_date"]

# Identify customers whose most recent purchase is more than 90 days before max_date
result_df = sales_df_transformed.alias("s") \
    .join(cust_df.alias("c"), col("s.customer_id") == col("c.customer_id"), "inner") \
    .groupBy(col("s.customer_id")) \
    .agg(max(col("s.invoice_date")).alias("latest_invoice_date")) \
    .filter(datediff(lit(max_date), col("latest_invoice_date")) > 90)

display(result_df)


## usecase 10

Create a clean, analysis-ready master dataset by joining customer and transaction tables. Derive revenue, month, day-of-week, and age group. Handle nulls. Persist as a Parquet table.

In [0]:
#result_df=
spark.sql("""SELECT 
    CASE 
        WHEN COALESCE(c.age,0) < 20 THEN 'Teenager'
        WHEN COALESCE(c.age,0) BETWEEN 20 AND 35 THEN 'Young Adults'
        WHEN COALESCE(c.age,0) BETWEEN 36 AND 50 THEN 'Adults'
        ELSE 'Senior'
    END AS age_group,
    MONTH(s.invoice_date) AS month,
    DAYOFWEEK(s.invoice_date) AS day_of_week,
    SUM(COALESCE(s.price,0) * COALESCE(s.quantity,0)) AS total_revenue
FROM cust_vw c
INNER JOIN sales_vw s 
    ON c.customer_id = s.customer_id
GROUP BY 
    age_group,
    MONTH(s.invoice_date),
    DAYOFWEEK(s.invoice_date)
ORDER BY 
    month,
    day_of_week""").write.mode("overwrite").parquet("/Volumes/izwd37dev/wd37db/rawdatta/BB2/result_Use_cases/")
#display(result_df)  
#result_df.write.mode("overwrite").parquet("/Volumes/izwd37dev/wd37db/rawdatta/BB2/result_Use_cases/")

#DSL

cust_df.alias("c") \
    .join(sales_df_transformed.alias("s"),
          col("c.customer_id") == col("s.customer_id"), "inner") \
    .withColumn(
        "age_group",
        when(coalesce(col("c.age"), lit(0)) < 20, "Teenager")
        .when((coalesce(col("c.age"), lit(0)) >= 20) & (coalesce(col("c.age"), lit(0)) <= 35), "Young Adults")
        .when((coalesce(col("c.age"), lit(0)) >= 36) & (coalesce(col("c.age"), lit(0)) <= 50), "Adults")
        .otherwise("Senior")
    ) \
    .groupBy(
        col("age_group"),
        month(col("s.invoice_date")).alias("month"),
        dayofweek(col("s.invoice_date")).alias("day_of_week")
    ) \
    .agg(
        sum(coalesce(col("s.price"), lit(0)) * coalesce(col("s.quantity"), lit(0))).alias("total_revenue")
    ) \
    .orderBy("month", "day_of_week") \
    .write.mode("overwrite") \
    .parquet("/Volumes/izwd37dev/wd37db/rawdatta/BB2/result_Use_cases/")


## usecase 11

What is the gender distribution across different product categories

In [0]:
spark.sql("select gender,category, count(1) gen_across_cat from cust_vw c inner join sales_vw s on c.customer_id=s.customer_id group by gender,category order by 2 ,3 desc,1").show()

cust_df.alias("c").join(sales_df_transformed.alias("s"),col("c.customer_id")==col("s.customer_id"),'inner').groupBy(col("c.gender"),col("s.category")).agg(count("*").alias("gen_across_cat")).orderBy(col("s.category"),col("gen_across_cat").desc(),col("c.gender")).show()

## usecase 12

What is the total revenue generated in the year 2022

In [0]:
spark.sql("select sum(price*quantity) Tot_revenue from sales_vw s inner join cust_vw c on s.customer_id=c.customer_id where year(s.invoice_date)='2022'").show()

#spark.sql("select year(s.invoice_date) yr, invoice_date from sales_vw s inner join cust_vw c on s.customer_id=c.customer_id where year(s.invoice_date)='2022'").show()
#spark.sql("select * from sales_vw s inner join cust_vw c on s.customer_id=c.customer_id where year(s.invoice_date)='2022'").show()

spark.sql("select year(s.invoice_date) yr, invoice_date from sales_vw s  where year(invoice_date)='2022'").count()

cust_df.alias("c").join(sales_df_transformed.alias("s"),col("c.customer_id")==col("s.customer_id"),"inner").filter(date_format(col("s.invoice_date"),"yyyy")=="2022").agg(sum(col("s.price")*col("s.quantity")).alias("Tot_revenue")).orderBy(col("Tot_revenue")).show()